# ML-09 — Validation and Research Claim Audit

**Lane:** Refresh / Content Opportunity Scoring.

This notebook reads two findings from *The State of AI-Driven SEO* (FlyRank, March 2026) constructively, then applies the same scrutiny to my Week-5 model. All results are public-safe, observational, and intended for decision support.

> Repository guidance used: `hunting-leakage-and-validating` + `flyrank/flyrank-data`.

## 1. Two paper findings and my methodology questions

### Finding 1 — growing content is longer and younger

The paper reports that the observed growing cohort averages about 3,180 words and 184 days old, versus 2,311 words and 230 days for the declining cohort (Finding #1, p. 6). It also says the comparison is observational.

**Constructive methodology question:** The label is described as the last 30 days of impressions versus the previous 30 days. Were the two cohorts compared within the same client and content-type distributions, and how sensitive are the word-count and age gaps to client weighting or a client-grouped resample? That would show whether the measured pattern generalizes across brands rather than being driven by a few large portfolios. The current comparison supports a directional association; it does not by itself show that adding words causes growth.

### Finding 2 — recently refreshed mature pages show much stronger measured outcomes

The paper reports a 3.2x health-score difference and 57x impression difference for 365+ day content refreshed within 30 days compared with older stale content (Finding #4, p. 9). It carefully calls the evidence observational elsewhere in the paper.

**Constructive methodology question:** Was the refresh date strictly before the performance window, and were refreshed pages matched with comparable unrefreshed pages on prior visibility, client, topic, and age? If refresh selection and the measured 90-day performance window overlap, strong pages may be more likely to receive a refresh, or the feature may partly contain the outcome period. A time-aware matched comparison would better support a refresh-effect claim; without it, the result is a strong measured association and a useful review hypothesis.

In [1]:
import pandas as pd

paper_findings = pd.DataFrame([
    {
        "paper_finding": "Growing pages are longer and younger",
        "reported_numbers": "3,180 vs 2,311 words; 184 vs 230 days",
        "methodology_question": "Does the pattern persist within clients/content types under grouped resampling?",
        "safe_read": "Observed directional association; not a word-count intervention effect.",
    },
    {
        "paper_finding": "Recently refreshed mature pages show stronger outcomes",
        "reported_numbers": "3.2x health; 57x impressions",
        "methodology_question": "Did refresh strictly precede outcomes, with matched untreated pages?",
        "safe_read": "Measured refresh association and review hypothesis; not causal proof.",
    },
])
display(paper_findings)

,paper_finding,reported_numbers,methodology_question,safe_read
0,Growing pages are longer and younger,"3,180 vs 2,311 words; 184 vs 230 days",Does the pattern persist within clients/conten...,Observed directional association; not a word-c...
1,Recently refreshed mature pages show stronger ...,3.2x health; 57x impressions,"Did refresh strictly precede outcomes, with ma...",Measured refresh association and review hypoth...


## 2. My model under an honest split: before and after

The **before** design is a stratified random row split. It is tempting because it preserves the overall label balance, but the same pseudonymized clients occur in both train and test, allowing shared client patterns to cross the boundary.

The **after** design holds out entire pseudonymized clients with `GroupShuffleSplit`. It asks the more useful deployment question: does the ranking transfer to clients the model has never seen?

Both versions use the frozen Week-4 candidate definition, the same 18 honest features, the same Random Forest settings, the same observed decline proxy, and the same metrics. Precision@50 remains the primary operational measure. The two test populations have different base rates, so ROC AUC and average precision are also shown; this before/after is a validation-design sensitivity check, not a controlled estimate of a single treatment effect.

In [2]:
from pathlib import Path
import json
import numpy as np
import sklearn
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import average_precision_score, roc_auc_score
from sklearn.model_selection import GroupShuffleSplit, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

SEED = 42
repo_root = Path.cwd()
while not (repo_root / "data" / "raw" / "content_refresh_anonymized.csv").exists():
    if repo_root == repo_root.parent:
        raise FileNotFoundError("Could not locate the starter dataset.")
    repo_root = repo_root.parent

df = pd.read_csv(repo_root / "data" / "raw" / "content_refresh_anonymized.csv")
df["is_declining_proxy"] = df["trend_direction"].eq("down").astype("int8")

eligible = (
    df["impressions_90d"].ge(300)
    & df["avg_position"].gt(0)
    & df["avg_position"].le(20)
    & df["ctr"].lt(0.50)
)
frame = df.loc[eligible].copy().reset_index(drop=True)
target = "is_declining_proxy"

numeric_features = [
    "search_volume", "competition", "cpc", "word_count",
    "content_age_days", "days_since_last_update",
    "impressions_90d", "clicks_90d", "sessions_90d",
    "days_with_impressions", "days_with_sessions",
    "ctr", "avg_position", "engagement_rate",
]
categorical_features = [
    "content_type", "main_intent", "competition_level", "freshness_tier",
]
feature_columns = numeric_features + categorical_features

def build_pipeline(numeric_columns):
    preprocessor = ColumnTransformer([
        ("numeric", Pipeline([
            ("impute", SimpleImputer(strategy="median", add_indicator=True)),
            ("scale", StandardScaler()),
        ]), numeric_columns),
        ("categorical", Pipeline([
            ("impute", SimpleImputer(strategy="most_frequent")),
            ("one_hot", OneHotEncoder(handle_unknown="ignore")),
        ]), categorical_features),
    ])
    forest = RandomForestClassifier(
        n_estimators=400,
        max_depth=8,
        min_samples_leaf=20,
        max_features="sqrt",
        class_weight="balanced_subsample",
        random_state=SEED,
        n_jobs=-1,
    )
    return Pipeline([("preprocess", preprocessor), ("model", forest)])

def precision_at_k(y, score, k):
    order = np.argsort(-np.asarray(score), kind="stable")
    return float(np.asarray(y)[order[:k]].mean())

def fit_and_measure(train_idx, test_idx, numeric_columns=numeric_features):
    columns = numeric_columns + categorical_features
    train = frame.iloc[train_idx]
    test = frame.iloc[test_idx]
    pipeline = build_pipeline(numeric_columns)
    pipeline.fit(train[columns], train[target])
    score = pipeline.predict_proba(test[columns])[:, 1]
    metrics = {
        "test_rows": int(len(test)),
        "test_base_rate": float(test[target].mean()),
        "precision_at_10": precision_at_k(test[target], score, 10),
        "precision_at_50": precision_at_k(test[target], score, 50),
        "precision_at_100": precision_at_k(test[target], score, 100),
        "roc_auc": float(roc_auc_score(test[target], score)),
        "average_precision": float(average_precision_score(test[target], score)),
    }
    return pipeline, test.copy(), score, metrics

all_idx = np.arange(len(frame))
random_train_idx, random_test_idx = train_test_split(
    all_idx,
    test_size=0.25,
    stratify=frame[target],
    random_state=SEED,
)
group_splitter = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=SEED)
group_train_idx, group_test_idx = next(
    group_splitter.split(frame, groups=frame["client_id"])
)

random_model, random_test, random_score, random_metrics = fit_and_measure(
    random_train_idx, random_test_idx
)
group_model, group_test, group_score, group_metrics = fit_and_measure(
    group_train_idx, group_test_idx
)

random_overlap = len(
    set(frame.iloc[random_train_idx]["client_id"])
    & set(frame.iloc[random_test_idx]["client_id"])
)
group_overlap = len(
    set(frame.iloc[group_train_idx]["client_id"])
    & set(frame.iloc[group_test_idx]["client_id"])
)

comparison = pd.DataFrame([
    {
        "validation": "Before: random rows",
        "client_overlap": random_overlap,
        **random_metrics,
    },
    {
        "validation": "After: grouped clients",
        "client_overlap": group_overlap,
        **group_metrics,
    },
])
metric_cols = [
    "test_base_rate", "precision_at_10", "precision_at_50",
    "precision_at_100", "roc_auc", "average_precision",
]
comparison[metric_cols] = comparison[metric_cols].round(3)

print("Frozen candidate population:", f"{len(frame):,}")
display(comparison)
print(
    "Measured grouped-minus-random ROC AUC gap:",
    f"{group_metrics['roc_auc'] - random_metrics['roc_auc']:+.3f}",
)
print(
    "Measured grouped-minus-random Precision@50 gap:",
    f"{group_metrics['precision_at_50'] - random_metrics['precision_at_50']:+.1%}",
)
print("Grouped client overlap:", group_overlap)

Frozen candidate population: 10,730


,validation,client_overlap,test_rows,test_base_rate,precision_at_10,precision_at_50,precision_at_100,roc_auc,average_precision
0,Before: random rows,25,2683,0.635,1.0,0.94,0.9,0.698,0.789
1,After: grouped clients,0,1701,0.554,1.0,0.78,0.8,0.639,0.680


Measured grouped-minus-random ROC AUC gap: -0.059
Measured grouped-minus-random Precision@50 gap: -16.0%
Grouped client overlap: 0


### Reading the before/after

The random split produces a stronger-looking result because pages from 25 clients appear on both sides. Under the client-grouped split, ROC AUC falls from about 0.698 to 0.639 and Precision@50 falls from 94% to 78%. Some of that difference reflects the grouped test cohort's lower measured base rate (55.4% versus 63.5%), and some is consistent with client-specific patterns making random validation optimistic.

The grouped result is the number I keep. It supports a directional ranking signal for unseen clients in this starter snapshot. It does not establish future performance because the file is a single trailing-window snapshot.

## 3. Leakage audit

### Timeline

`90-day snapshot features ── overlap in calendar time ── observed 30d-vs-prev-30d proxy`

This timeline exposes the largest remaining limitation: even after removing the explicit 30-day comparison columns, several 90-day aggregates overlap the windows used to construct the proxy. They are available in the snapshot but are **not strictly prior** to the outcome. Therefore this notebook describes cross-client discrimination inside one snapshot, not a future forecast.

### Feature checks

- **Label-derived siblings:** `trend_direction` and `trend_pct` are forbidden. The six last/previous-30-day columns are also forbidden.
- **Identifiers:** `content_id` and `client_id` are never features; `client_id` is used only for grouping.
- **Decision-derived inputs:** no product flags, existing scores, reason codes, provider labels, or model labels enter the model.
- **Systematic missingness:** median/category imputation is fitted on training rows only, with numeric missingness indicators.
- **Population definition:** eligibility uses 90-day impressions, position, and CTR. It does not use the label directly, but it overlaps the label's measurement period and is disclosed as a limitation.

The deliberate trap below adds `trend_pct`, the numeric source of the target, to confirm the audit harness catches leakage. The near-perfect result is rejected; the honest grouped result remains the reported result.

In [3]:
forbidden_features = {
    "content_id", "client_id", "trend_direction", "trend_pct",
    "is_declining_proxy",
    "impressions_last_30d", "clicks_last_30d", "sessions_last_30d",
    "impressions_prev_30d", "clicks_prev_30d", "sessions_prev_30d",
    "provider_used", "model_used",
}
honest_overlap = sorted(set(feature_columns) & forbidden_features)

leaky_numeric = numeric_features + ["trend_pct"]
_, leaky_test, leaky_score, leaky_metrics = fit_and_measure(
    group_train_idx, group_test_idx, numeric_columns=leaky_numeric
)

leakage_comparison = pd.DataFrame([
    {"feature_set": "Rejected: includes trend_pct", **leaky_metrics},
    {"feature_set": "Kept: honest Week-5 features", **group_metrics},
])
leakage_comparison[metric_cols] = leakage_comparison[metric_cols].round(3)

print("Honest feature overlap with forbidden fields:", honest_overlap)
display(leakage_comparison[[
    "feature_set", "test_base_rate", "precision_at_50",
    "roc_auc", "average_precision",
]])
print("Leak test verdict: CAUGHT AND REMOVED")

assert honest_overlap == []
assert leaky_metrics["roc_auc"] > 0.99
assert group_metrics["roc_auc"] < leaky_metrics["roc_auc"]

Honest feature overlap with forbidden fields: []


,feature_set,test_base_rate,precision_at_50,roc_auc,average_precision
0,Rejected: includes trend_pct,0.554,1.00,1.000,1.00
1,Kept: honest Week-5 features,0.554,0.78,0.639,0.68


Leak test verdict: CAUGHT AND REMOVED


### Real grouped-holdout failure examples

The error examples are high-ranked pages whose observed proxy is not decline. They may still warrant review; they are wrong relative to this target. Missing query mix, SERP features, seasonality, and edit timing can all make an apparently risky page behave differently.

In [4]:
group_test["model_score"] = group_score
ranked_group_test = group_test.sort_values(
    ["model_score", "content_id"],
    ascending=[False, True],
    kind="stable",
).reset_index(drop=True)
ranked_group_test["model_rank"] = np.arange(1, len(ranked_group_test) + 1)

wrong_cases = ranked_group_test.loc[ranked_group_test[target].eq(0)].head(3).copy()
wrong_cases["audit_note"] = [
    "Measured exposure and activity resemble decline cases, but this page's observed proxy is not down.",
    "The score may be reacting to stale-looking aggregate signals while missing query or seasonal context.",
    "A cross-client pattern ranks this page highly, but the snapshot cannot show whether a recent edit changed its state.",
]
display(wrong_cases[[
    "model_rank", "content_id", "model_score", "impressions_90d",
    "ctr", "avg_position", "days_since_last_update", "audit_note",
]].round({"model_score": 3, "ctr": 3, "avg_position": 1}))

error_summary = pd.DataFrame({
    "measured_quantity": [
        "Top-50 false positives",
        "Top-100 false positives",
        "All-test ROC AUC",
    ],
    "value": [
        int((ranked_group_test.head(50)[target] == 0).sum()),
        int((ranked_group_test.head(100)[target] == 0).sum()),
        round(group_metrics["roc_auc"], 3),
    ],
})
display(error_summary)

,model_rank,content_id,model_score,impressions_90d,ctr,avg_position,days_since_last_update,audit_note
10,11,content_b8fe98256cb9,0.717,1623,0.18,7.6,104,Measured exposure and activity resemble declin...
18,19,content_d788e2c18ae8,0.702,1377,0.07,11.4,104,The score may be reacting to stale-looking agg...
21,22,content_502b28c930ff,0.700,866,0.00,6.1,20,"A cross-client pattern ranks this page highly,..."


,measured_quantity,value
0,Top-50 false positives,11.000
1,Top-100 false positives,20.000
2,All-test ROC AUC,0.639


## 4. Claim rewrite

**Too strong:** “The Random Forest predicts which pages will decline and proves that these pages should be refreshed.”

**Rewritten:** “On a client-grouped holdout from the public starter snapshot, the Random Forest ranked the observed decline proxy above the 55.4% test base rate, reaching 78% Precision@50 and ROC AUC 0.639. The measured result is directional and supports a human review queue for unseen clients in this sample. Because the 90-day features overlap the proxy's comparison windows and no refresh intervention was tested, it does not establish future prediction or causal refresh impact.”

**Action boundary:** use the score to decide what to inspect first. Before changing content, review current query mix, page state, seasonality, and business relevance.

In [5]:
claim_audit = pd.DataFrame([
    {
        "claim_element": "Observed population",
        "safe_statement": f"{len(group_test):,} eligible rows from {group_test['client_id'].nunique()} unseen pseudonymized clients",
    },
    {
        "claim_element": "Measured ranking result",
        "safe_statement": (
            f"Precision@50 {group_metrics['precision_at_50']:.0%}; "
            f"ROC AUC {group_metrics['roc_auc']:.3f}; "
            f"base rate {group_metrics['test_base_rate']:.1%}"
        ),
    },
    {
        "claim_element": "Decision supported",
        "safe_statement": "Prioritize manual review; do not auto-edit.",
    },
    {
        "claim_element": "Cannot claim",
        "safe_statement": "Future prediction, Google's mechanism, or causal refresh impact.",
    },
])
display(claim_audit)

receipt = {
    "lane": "Refresh / Content Opportunity Scoring",
    "random_seed": SEED,
    "before_random_split": random_metrics,
    "before_client_overlap": random_overlap,
    "after_grouped_split": group_metrics,
    "after_client_overlap": group_overlap,
    "leaky_trend_pct_test": leaky_metrics,
    "honest_feature_columns": feature_columns,
    "forbidden_feature_overlap": honest_overlap,
    "error_example_count": int(len(wrong_cases)),
    "sklearn_version": sklearn.__version__,
    "claim_scope": "observed, measured, directional, decision-support",
    "timeline_limitation": (
        "Starter 90-day aggregates overlap the 30d-vs-prev30d proxy; "
        "this is not a future-window test."
    ),
}
receipt_path = repo_root / "work" / "outputs" / "w06_validation_metrics.json"
receipt_path.parent.mkdir(parents=True, exist_ok=True)
receipt_path.write_text(json.dumps(receipt, indent=2) + "\n")

assert group_overlap == 0
assert random_overlap > 0
assert len(wrong_cases) == 3
assert honest_overlap == []
assert receipt_path.exists()

print("Wrote:", receipt_path.relative_to(repo_root))
print("Validation audit: PASS")

,claim_element,safe_statement
0,Observed population,"1,701 eligible rows from 7 unseen pseudonymize..."
1,Measured ranking result,Precision@50 78%; ROC AUC 0.639; base rate 55.4%
2,Decision supported,Prioritize manual review; do not auto-edit.
3,Cannot claim,"Future prediction, Google's mechanism, or caus..."


Wrote: work/outputs/w06_validation_metrics.json
Validation audit: PASS


## 5. Self-check

- [x] Two paper findings are named with exact reported numbers and constructive methodology questions.
- [x] The same model is shown before (random rows) and after (client-grouped holdout).
- [x] Base rates, Precision@K, ROC AUC, and average precision are visible.
- [x] The grouped split has zero client overlap and is the result retained.
- [x] The feature timeline and overlapping-window limitation are explicit.
- [x] Label-derived, decision-derived, ID, and recent-window fields are excluded.
- [x] A deliberate leaky feature is caught, rejected, and removed.
- [x] Three real pseudonymized grouped-holdout errors are inspected.
- [x] The bold claim is rewritten as observed, measured, directional decision support.
- [x] No client names, domains, URLs, private queries, credentials, or raw exports appear.
- [x] The notebook runs top to bottom with visible outputs.